# Phase 2 — Member 1: Simple MLP Baseline for Flight Delay Prediction

This notebook demonstrates the complete implementation, training results, and evaluation for **Member 1: Simple Multi-Layer Perceptron (MLP) Baseline using PyTorch and One-Hot Encoding**.

### Objectives:
1. Build a deep learning baseline for binary flight delay prediction (`is_delayed = 1 if arr_delay >= 15 else 0`).
2. Preprocess 59 engineered features (49 numerical standardized + 10 categorical one-hot encoded -> 1,015 input dimensions).
3. Train PyTorch MLP baseline with Adam, BCEWithLogitsLoss, and early stopping on validation PR-AUC.
4. Evaluate on the September-October 2013 validation set across PR-AUC, ROC-AUC, Brier score, and ECE.
5. Compare with Phase 1 baselines and demonstrate reusable inference.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from flight_delay.config import load_config
from flight_delay.features.build import CATEGORICAL, numeric_features
from flight_delay.models.mlp import MLPBaseline
from flight_delay.models.mlp_training import MLPPreprocessor, predict_flight_delays, evaluate_mlp, get_device

cfg = load_config(ROOT / 'configs' / 'config.yaml')
print('PyTorch version:', torch.__version__)
print('Device detected:', get_device())
print('Seed configured:', cfg['seed'])

## 1. Dataset & Preprocessing Pipeline

The chronological splits are strictly preserved:
* **Training**: January 1 – August 31, 2013 (217,727 flights, 26.37% delayed)
* **Validation**: September 1 – October 31, 2013 (55,628 flights, 15.70% delayed)
* **Test (Sealed)**: November 1 – December 31, 2013 (Never loaded or evaluated in dev)

In [ ]:
model_dir = ROOT / 'models' / 'phase2' / 'member1'
with open(model_dir / 'run_metadata.json') as f:
    metadata = json.load(f)

print('Total Training Rows:', f"{metadata['training_rows']:,}")
print('Total Validation Rows:', f"{metadata['validation_rows']:,}")
print('Numerical Features:', len(metadata['numeric_features']))
print('Categorical Features:', len(metadata['categorical_features']))
print('One-Hot Features Generated:', metadata['one_hot_feature_count'])
print('Actual Total MLP Input Dimension:', metadata['total_input_dim'])

## 2. MLP Baseline Architecture

Architecture:
Input (1015) -> Linear(128) -> ReLU -> Dropout(0.3) -> Linear(64) -> ReLU -> Dropout(0.2) -> Linear(1 (Logits))

* No sigmoid in forward method (for numerical stability with BCEWithLogitsLoss).
* Sigmoid applied during probability prediction and metric evaluation.

In [ ]:
arch_fig_path = ROOT / 'reports' / 'figures' / 'phase2_member1' / 'mlp_architecture.png'
display(Image(filename=str(arch_fig_path)))

## 3. Training History & Early Stopping

Training configuration:
* Optimizer: Adam (lr = 0.001)
* Batch size: 256
* Loss: BCEWithLogitsLoss
* Early stopping: Patience 5 on Validation PR-AUC

In [ ]:
with open(model_dir / 'training_history.json') as f:
    history = json.load(f)

history_df = pd.DataFrame(history)
display(history_df)

fig_loss = ROOT / 'reports' / 'figures' / 'phase2_member1' / 'loss_curve.png'
fig_pr = ROOT / 'reports' / 'figures' / 'phase2_member1' / 'valid_pr_auc_vs_epoch.png'
display(Image(filename=str(fig_loss)))
display(Image(filename=str(fig_pr)))

## 4. Validation Performance & Evaluation Metrics

Evaluated on the September–October 2013 validation set (55,628 flights).

In [ ]:
with open(model_dir / 'validation_metrics.json') as f:
    metrics = json.load(f)

print('=== Validation Set Performance (Sept-Oct 2013) ===')
print(f"1. PR-AUC (Average Precision): {metrics['pr_auc']:.5f} (Primary Metric)")
print(f"   Validation Delay Rate Baseline: {metrics['positive_rate']:.5f}")
print(f"   PR-AUC Lift over Baseline:     +{metrics['pr_auc'] - metrics['positive_rate']:.5f}")
print(f"2. ROC-AUC:                    {metrics['roc_auc']:.5f}")
print(f"3. Brier Score:                 {metrics['brier_score']:.5f}")
print(f"4. Expected Calibration Error:  {metrics['ece_10']:.5f}")
print(f"5. Validation BCE Loss:         {metrics['loss']:.5f}")
print(f"6. Precision (thresh=0.50):     {metrics['precision']:.5f}")
print(f"7. Recall (thresh=0.50):        {metrics['recall']:.5f}")
print(f"8. F1-Score (thresh=0.50):      {metrics['f1_score']:.5f}")
print(f"9. Confusion Matrix:           {metrics['confusion_matrix']}")

In [ ]:
fig_cm = ROOT / 'reports' / 'figures' / 'phase2_member1' / 'confusion_matrix.png'
fig_pr_curve = ROOT / 'reports' / 'figures' / 'phase2_member1' / 'precision_recall_curve.png'
display(Image(filename=str(fig_cm)))
display(Image(filename=str(fig_pr_curve)))

## 5. Comparison with Phase 1 Baselines

Note on evaluation protocol context:
* Phase 1 Logistic Regression and XGBoost were tuned on Jan–Jul and selected on August 2013 (high-delay summer period, rate ~28%).
* Phase 2 MLP Baseline is evaluated on the September–October validation set (lower-delay autumn period, rate = 15.70%).

In [ ]:
comp_df = pd.read_csv(ROOT / 'reports' / 'results' / 'phase2_member1' / 'model_comparison.csv')
display(comp_df)

## 6. End-to-End Reusable Inference Demonstration

Demonstrating the reusable `predict_flight_delays()` function that loads the saved preprocessing artifacts and PyTorch weights to score new flights.

In [ ]:
sample_df = pd.read_parquet(ROOT / 'data' / 'processed' / 'valid.parquet').head(10)
probs, preds = predict_flight_delays(
    artifacts_dir=model_dir,
    df=sample_df,
    threshold=0.50,
    device='cpu',
)

results_sample = pd.DataFrame({
    'id': sample_df['id'],
    'carrier': sample_df['carrier'],
    'origin': sample_df['origin'],
    'dest': sample_df['dest'],
    'true_delayed': sample_df['is_delayed'],
    'pred_probability': probs,
    'pred_label': preds,
})
display(results_sample)